# 01 · Dados da bacia do Rio Tubarão e análise de frequência de cheias

**Objetivo:** baixar as séries históricas da ANA (nível, vazão e chuva) das estações da bacia,
checar a qualidade e fazer a primeira estimativa de **tempo de retorno** das cheias.

> ⚠️ **Projeto experimental.** Nada aqui substitui os alertas oficiais da Defesa Civil,
> do SGB/CPRM e da ANA.

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lucaswsiviero/tubarao-flood/blob/main/notebooks/01_dados_bacia.ipynb)

**Como usar no Colab:** `Ambiente de execução > Executar tudo`. Leva alguns minutos, porque o
servidor da ANA é lento. Os dados baixados ficam em cache, então a segunda execução é rápida.

## 1. Preparação

In [ ]:
import os, sys
from pathlib import Path

EM_COLAB = "google.colab" in sys.modules
if EM_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")           # pede autorização uma vez
    DATA_DIR = Path("/content/drive/MyDrive/tubarao-flood/data")
else:
    DATA_DIR = Path("data")
(DATA_DIR / "raw").mkdir(parents=True, exist_ok=True)
print("Dados serão salvos em:", DATA_DIR.resolve())

O código de acesso à ANA (serviço público, sem login) fica no repositório do GitHub, em
`src/tubarao/ana.py`. A célula abaixo clona (ou atualiza) o repositório e importa o módulo.

In [ ]:
import subprocess, sys
REPO = "https://github.com/lucaswsiviero/tubarao-flood.git"
PASTA = Path("tubarao-flood")
if PASTA.exists():
    try:
        subprocess.run(["git", "-C", str(PASTA), "pull", "--ff-only"], check=False)   # pega atualizações
    except OSError:
        pass                                                                          # sem git: usa a cópia local
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, str(PASTA)], check=True)
sys.path.insert(0, str(PASTA / "src"))

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from tubarao import ana
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})
pd.options.display.float_format = "{:,.1f}".format

## 2. Estações da bacia

Consultamos o inventário da ANA para Santa Catarina e ficamos com as estações nos rios
da bacia do Rio Tubarão (Tubarão, Capivari, Braço do Norte e afluentes).

In [ ]:
RIOS = ["RIO TUBARÃO", "RIO CAPIVARI", "RIO BRAÇO DO NORTE", "RIO PEQUENO",
        "RIO BRAÇO ESQUERDO", "RIO POVOAMENTO", "RIO D'UNA"]
inv_f = ana.inventario("fluviometrica")
inv_p = ana.inventario("pluviometrica")

bacia_f = inv_f[inv_f.RioNome.isin(RIOS)].copy()
print(f"{len(bacia_f)} estações fluviométricas nos rios da bacia")
bacia_f[["Codigo", "Nome", "RioNome", "nmMunicipio", "Latitude", "Longitude"]].sort_values(["RioNome", "Codigo"]).reset_index(drop=True)

In [ ]:
# Mapa simples (lat/lon) das estações fluviométricas e pluviométricas da região
caixa = dict(lat=(-28.75, -27.85), lon=(-49.45, -48.65))
p = inv_p[inv_p.Latitude.between(*caixa["lat"]) & inv_p.Longitude.between(*caixa["lon"])]
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(p.Longitude, p.Latitude, s=14, c="tab:blue", label=f"pluviométricas ({len(p)})")
ax.scatter(bacia_f.Longitude, bacia_f.Latitude, s=22, c="tab:red", label=f"fluviométricas ({len(bacia_f)})")
ax.scatter([-48.97], [-28.47], marker="*", s=250, c="gold", edgecolor="k", label="Tubarão (cidade)")
ax.set_xlabel("longitude"); ax.set_ylabel("latitude"); ax.legend(); ax.set_title("Estações ANA na região")
plt.show()

## 3. Download das séries-chave

Escolhemos estações com histórico longo, de montante para jusante:
**Orleans** e **Pedras Grandes** (Rio Tubarão), **Armazém** (Rio Capivari), **São Ludgero**
(Rio Braço do Norte) e **Tubarão** (cidade), mais pluviômetros espalhados pela bacia.
Se alguma estação vier vazia, ela é ignorada.

In [ ]:
ALVOS = {
    # nome legível: (código, tipo)
    "Tubarão · cota":               ("84580500", "cota"),
    "Pedras Grandes · vazão":       ("84300000", "vazao"),
    "Pedras Grandes · cota":        ("84300000", "cota"),
    "Orleans II · vazão":           ("84250000", "vazao"),
    "Armazém (Capivari) · vazão":   ("84600000", "vazao"),
    "Gravatal (Capivari) · vazão":  ("84610000", "vazao"),
    "São Ludgero (B. Norte) · vazão": ("84560000", "vazao"),
    "Chuva Tubarão":                ("2849027", "chuva"),
    "Chuva Pedras Grandes":         ("2849028", "chuva"),
    "Chuva Orleans":                ("2849093", "chuva"),
    "Chuva Braço do Norte":         ("2849030", "chuva"),
    "Chuva Armazém":                ("2848000", "chuva"),
}

def carregar(cod, tipo, forcar=False):
    arq = DATA_DIR / "raw" / f"{cod}_{tipo}.parquet"
    if arq.exists() and not forcar:
        return pd.read_parquet(arq)
    df = ana.serie_diaria(cod, tipo)
    if not df.empty:
        df.to_parquet(arq)
    return df

series = {}
for nome, (cod, tipo) in ALVOS.items():
    df = carregar(cod, tipo)
    if df.empty:
        print(f"⚠️  {nome} ({cod}): sem dados, ignorada")
        continue
    series[nome] = df
    print(f"✔ {nome:32s} {cod}  {df.data.min().date()} → {df.data.max().date()}  ({len(df):,} dias)")

## 4. Qualidade dos dados

Primeiro, onde há **falhas**? A tabela mostra a fração de dias com dado em cada mês.
Cores claras = lacunas. Isso decide quais anos podem entrar na análise de extremos.

In [ ]:
def mapa_cobertura(df, titulo, ax):
    d = df.set_index("data").valor
    cont = d.groupby([d.index.year, d.index.month]).count().unstack()
    dias = pd.Series(pd.period_range("2000-01", periods=12, freq="M").days_in_month, index=range(1, 13))
    anos = range(int(cont.index.min()), int(cont.index.max()) + 1)       # inclui anos SEM nenhum dado
    frac = cont.reindex(index=anos, columns=range(1, 13)).fillna(0).div(dias, axis=1).clip(upper=1)
    ax.imshow(frac.T.values, aspect="auto", cmap="Blues", vmin=0, vmax=1,
              extent=[frac.index.min() - .5, frac.index.max() + .5, 12.5, .5])
    ax.set_title(titulo, fontsize=9); ax.set_yticks(range(1, 13, 3)); ax.grid(False)

fluv = [n for n in series if "Chuva" not in n]
fig, axs = plt.subplots(len(fluv), 1, figsize=(10, 1.7 * len(fluv)), sharex=True)
axs = np.atleast_1d(axs)
for ax, n in zip(axs, fluv):
    mapa_cobertura(series[n], n, ax)
plt.tight_layout(); plt.show()

In [ ]:
resumo = []
for n, df in series.items():
    anos = df.groupby(df.data.dt.year).size()
    resumo.append(dict(serie=n, inicio=df.data.min().date(), fim=df.data.max().date(),
                       dias=len(df), anos_completos=int((anos >= 330).sum()),
                       pct_consistido=100 * (df.consistencia == 2).mean(),
                       maximo=df.valor.max(), data_max=df.loc[df.valor.idxmax(), "data"].date()))
pd.DataFrame(resumo).set_index("serie")

In [ ]:
fig, axs = plt.subplots(len(fluv), 1, figsize=(10, 1.9 * len(fluv)), sharex=True)
axs = np.atleast_1d(axs)
for ax, n in zip(axs, fluv):
    d = series[n].set_index("data").valor.asfreq("D")      # dias sem dado viram buracos (NaN)
    ax.plot(d.index, d.values, lw=.5)
    ax.set_title(n, fontsize=9, loc="left")
plt.tight_layout(); plt.show()

### 4.1 Verificação de "mudança de referência" nas cotas

Cota é a altura da água numa **régua**. Se a régua é trocada ou o zero muda, a série dá um "salto"
e deixa de ser comparável. Abaixo, o **nível-base de cada ano** (percentil 10, ou seja, o rio
em águas baixas). Um salto repentino indica troca de referência.

In [ ]:
def anos_suspeitos(df, tol=40):
    """Anos cujo nível-base (p10) foge mais de `tol` da mediana dos anos."""
    p10 = df.groupby(df.data.dt.year).valor.quantile(.1)
    return p10[(p10 - p10.median()).abs() > tol]

cotas = [n for n in series if "cota" in n]
fig, axs = plt.subplots(len(cotas), 1, figsize=(9, 2.4 * len(cotas)), sharex=True)
axs = np.atleast_1d(axs)
for ax, n in zip(axs, cotas):
    p10 = series[n].groupby(series[n].data.dt.year).valor.quantile(.1)
    ax.plot(p10.index, p10.values, "o-", ms=3)
    ax.set_title(f"{n}: nível-base anual (p10, cm)", fontsize=9, loc="left")
    s = anos_suspeitos(series[n])
    if len(s):
        ax.plot(s.index, s.values, "ro", ms=5, label="anos suspeitos"); ax.legend()
plt.tight_layout(); plt.show()

SUSPEITOS = {n: sorted(anos_suspeitos(series[n]).index) for n in cotas}
for n, a in SUSPEITOS.items():
    print(f"{n}: anos suspeitos = {a if a else 'nenhum'}")

## 5. Maiores eventos registrados

Lista os maiores picos de cada série, **separados por pelo menos 10 dias** (para um mesmo evento
não aparecer várias vezes). Quando datas coincidem em várias estações, é um evento regional.

In [ ]:
def top_eventos(df, n=8, separacao=10):
    d = df.sort_values("valor", ascending=False)
    escolhidos = []
    for _, r in d.iterrows():
        if all(abs((r.data - e.data).days) >= separacao for e in escolhidos):
            escolhidos.append(r)
        if len(escolhidos) == n:
            break
    return pd.DataFrame(escolhidos)[["data", "valor"]].reset_index(drop=True)

for n in ["Pedras Grandes · vazão", "Armazém (Capivari) · vazão", "Tubarão · cota", "Chuva Tubarão"]:
    if n in series:
        print(f"\n=== {n} ===")
        display(top_eventos(series[n]).assign(data=lambda x: x.data.dt.date))

## 6. Análise de frequência (tempo de retorno)

Método clássico: pegamos o **máximo anual** de cada ano com dados suficientes e ajustamos uma
distribuição **GEV** (valores extremos generalizados). Dela sai o **tempo de retorno (TR)**:
um evento de TR = 50 anos tem 2% de chance de ocorrer em um dado ano.

**Como ajustamos:** com **momentos-L** (Hosking), que são estáveis com poucos anos de dados, e com o
parâmetro de forma `k` limitado a [-0,4; 0,4]. (A versão anterior usava máxima verossimilhança sem
limite, e com 27–37 pontos o ajuste "explodia": TR de 100 anos virava 10¹³.)
`k < 0` significa cauda pesada (cheias extremas mais prováveis); `k > 0`, cauda limitada.

**Limitações importantes (ficam no relatório final):**
- O máximo da *média diária* é menor que o pico instantâneo.
- Com 30–40 anos de dados, TR acima de ~50 anos é muito incerto (veja as faixas).
- Os picos de vazão acima do que já foi medido são **extrapolação da curva-chave** da estação.
- Pressupõe clima estacionário, o que é discutível.
- Anos com mudança de referência (seção 4.1) são excluídos.

In [ ]:
from math import gamma

def maximos_anuais(df, min_dias=300, excluir=()):
    g = df.groupby(df.data.dt.year)
    ok = (g.size() >= min_dias) & (~g.size().index.isin(list(excluir)))
    return g.valor.max()[ok]

# ---- GEV por momentos-L (Hosking 1990). Convenção: F(x)=exp(-(1-k(x-xi)/a)^(1/k)) ----
def momentos_l(x):
    x = np.sort(np.asarray(x, float)); n = len(x); j = np.arange(1, n + 1)
    b0 = x.mean()
    b1 = np.sum((j - 1) / (n - 1) * x) / n
    b2 = np.sum((j - 1) * (j - 2) / ((n - 1) * (n - 2)) * x) / n
    l1, l2, l3 = b0, 2 * b1 - b0, 6 * b2 - 6 * b1 + b0
    return l1, l2, l3 / l2

def gev_ajusta(x, k_lim=0.4):
    l1, l2, t3 = momentos_l(x)
    c = 2 / (3 + t3) - np.log(2) / np.log(3)
    k = float(np.clip(7.8590 * c + 2.9554 * c ** 2, -k_lim, k_lim))
    if abs(k) < 1e-6:
        a = l2 / np.log(2); xi = l1 - 0.5772156649 * a
    else:
        a = l2 * k / ((1 - 2 ** (-k)) * gamma(1 + k)); xi = l1 + a * (gamma(1 + k) - 1) / k
    return xi, a, k

def gev_nivel(T, par):
    xi, a, k = par
    y = -np.log(1 - 1 / np.asarray(T, float))
    return xi - a * np.log(y) if abs(k) < 1e-6 else xi + a / k * (1 - y ** k)

def gev_cdf(x, par):
    xi, a, k = par
    if abs(k) < 1e-6:
        return float(np.exp(-np.exp(-(x - xi) / a)))
    y = 1 - k * (x - xi) / a
    if y <= 0:
        return 1.0 if k > 0 else 0.0
    return float(np.exp(-y ** (1 / k)))

def ajusta_gev(x, trs=(2, 5, 10, 25, 50, 100), n_boot=1000, seed=0):
    par = gev_ajusta(x)
    rng = np.random.default_rng(seed)
    boots = np.array([gev_nivel(trs, gev_ajusta(rng.choice(x, size=len(x), replace=True)))
                      for _ in range(n_boot)])
    lo, hi = np.percentile(boots, [5, 95], axis=0)
    tab = pd.DataFrame({"TR (anos)": trs, "estimado": gev_nivel(trs, par), "IC90% inf": lo, "IC90% sup": hi})
    return par, tab

def grafico_gev(x, par, tab, titulo, unidade):
    xs = np.sort(x); n = len(xs)
    tr_emp = (n + 1) / (n + 1 - np.arange(1, n + 1))        # posições de plotagem de Weibull
    T = np.logspace(0.15, 2.2, 100)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.semilogx(T, gev_nivel(T, par), "k-", label="GEV ajustada (momentos-L)")
    ax.fill_between(tab["TR (anos)"], tab["IC90% inf"], tab["IC90% sup"], alpha=.2, label="IC 90% (bootstrap)")
    ax.semilogx(tr_emp, xs, "o", label="máximos anuais observados")
    ax.set_ylim(0, xs.max() * 1.8)
    ax.set_xlabel("tempo de retorno (anos)"); ax.set_ylabel(unidade)
    ax.set_title(f"{titulo}  ·  k = {par[2]:+.2f}"); ax.legend()
    plt.show()

ESTACOES_GEV = [("Pedras Grandes · vazão", "m³/s"), ("São Ludgero (B. Norte) · vazão", "m³/s"),
                ("Armazém (Capivari) · vazão", "m³/s"), ("Orleans II · vazão", "m³/s"),
                ("Tubarão · cota", "cm")]
RESULTADOS = {}
for nome, un in ESTACOES_GEV:
    if nome not in series:
        continue
    excl = SUSPEITOS.get(nome, [])
    x = maximos_anuais(series[nome], excluir=excl).values
    if len(x) < 15:
        print(f"{nome}: apenas {len(x)} anos utilizáveis, pulando")
        continue
    par, tab = ajusta_gev(x)
    RESULTADOS[nome] = (x, par, tab)
    print()
    print(f"=== {nome} · {len(x)} anos de máximos anuais" + (f" (excluídos por referência suspeita: {excl})" if excl else "") + " ===")
    display(tab.set_index("TR (anos)"))
    grafico_gev(x, par, tab, nome, un)

### Quanto vale o maior evento já registrado?
Pelo ajuste, qual o tempo de retorno do maior máximo anual observado? Se o valor sair muito alto,
ou o evento foi mesmo excepcional, ou a amostra é curta demais para saber. É por isso que o
histórico anterior a 1985 (jornais, relatórios da Defesa Civil) será importante.

In [ ]:
for nome, (x, par, tab) in RESULTADOS.items():
    pico = x.max()
    p_exced = 1 - gev_cdf(pico, par)
    tr = np.inf if p_exced <= 0 else 1 / p_exced
    print(f"{nome:32s} n={len(x):3d}  maior máximo anual = {pico:8.1f}  →  TR ≈ {tr:,.0f} anos  (k = {par[2]:+.2f})")

## 7. Salvar resultados

In [ ]:
OUT = DATA_DIR / "processed"; OUT.mkdir(exist_ok=True)
bacia_f.to_csv(OUT / "estacoes_fluviometricas_bacia.csv", index=False)
for nome, (x, par, tab) in RESULTADOS.items():
    tab.to_csv(OUT / f"gev_{nome.replace(' · ', '_').replace(' ', '_')}.csv", index=False)
print("Arquivos em", OUT.resolve()); print(*sorted(p.name for p in OUT.iterdir()), sep="\n")

## Próximos passos
1. **Catálogo de cheias**: juntar picos de vazão/cota + chuva dos 3–7 dias anteriores + registros
   de desastre (S2iD, Defesa Civil, jornais), incluindo eventos anteriores a 1985.
2. **Chuva antecedente**: baixar ERA5/CHIRPS da bacia e medir quanta chuva (e onde) antecede as cheias.
3. **Tempo de propagação** entre montante (Orleans/Pedras Grandes) e a cidade de Tubarão.
4. **Dados recentes de Tubarão** (a série histórica de cota termina em 2021): usar a telemetria da ANA/SGB.